# Построение производных признаков


Цель данного ноутбука - сформировать набор производных признаков, которые позволят более устойчиво анализировать цену, доступность, активность отзывов и различия между типами хостов.


## Загрузка библиотек и установка настроек


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:.4f}".format)


## Загрузка очищенного набора данных


In [2]:
listings = pd.read_parquet("../data/processed/listings_clean.parquet")


## Формирование производных признаков


### Порог для выявления выбросов
Для признака `price` используется межквартильный размах. Высокие значения далее будут интерпретироваться как выбросы.


In [ ]:
Q1 = listings["price"].quantile(0.25)
Q3 = listings["price"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR


### Признак `has_last_review`
Флаг отделяет объявления с историей отзывов от объектов без отзывной активности.


In [5]:
listings["has_last_review"] = listings["last_review"].notna()


### Признак `has_price`
Признак переводит информацию о пропуске цены в явный бинарный формат.


In [6]:
listings["has_price"] = listings["price"].notna()


### Признак `is_price_outlier`
После расчёта квартильной границы формируется индикатор экстремально дорогих объектов.


In [7]:
listings["is_price_outlier"] = listings["price"] > upper_bound


### Признак `host_size_segment`
Хосты делятся на три сегмента по числу активных объявлений: `Individual host`, `Experienced host` и `Industrial host`.

In [ ]:
bins = [1, 2, 5, 999999]
labels = ["Individual host", "Experienced host", "Industrial host"]

listings["host_size_segment"] = pd.cut(
    listings["calculated_host_listings_count"], bins=bins, labels=labels, right=False
)


### Признак `availability_ratio`
Абсолютное значение `availability_365` переводится в нормированную долю года в диапазоне от *0* до *1*.


In [9]:
listings["availability_ratio"] = listings["availability_365"] / 365


### Признак `review_intensity`
Показатель отражает, какая часть накопленных отзывов пришлась на последние 12 месяцев.


In [ ]:
listings["review_intensity"] = np.where(
    listings["number_of_reviews"] == 0,
    0,
    listings["number_of_reviews_ltm"] / listings["number_of_reviews"],
)


### Признак `log_price`
Логарифмирование `price` сглаживает длинный правый хвост и делает ценовой слой более удобным для визуального анализа.

In [11]:
listings["log_price"] = np.log1p(listings["price"])


## Сохранение изменений


In [12]:
listings.to_parquet("../data/processed/listings_analytics.parquet", index=False)


## Промежуточные выводы
В результате формируется аналитический набор данных, в котором исходные сведения дополнены бинарными индикаторами, сегментацией хостов и нормированными производными признаками.

1. Поля `has_last_review` и `has_price` переводят пропуски в интерпретируемые значения.
2. Признаки `is_price_outlier`, `availability_ratio`, `review_intensity` и `log_price` подготавливают основу для последующего анализа.
3. Сегментация `host_size_segment` позволяет сравнивать мелких и крупных хостов.
